## Задача: Банк теряет клиентов. Нужно предсказать, уйдет ли клиент (1) или останется (0).
## Бизнес-ценность: удержать старого клиента дешевле, чем привель нового. Бизнесу нужно знать вероятность ухода, чтобы звонить тем у кого она высокая


In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [2]:
df = pd.read_csv("D:\\ML\\ML\\ML_project\\archive\\WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [5]:
df.groupby('gender')['Churn'].value_counts() # тут решил просто посмотреть распределение данных по полу, не знаю зачем можно было и по колонке churn посчитать

gender  Churn
Female  No       2549
        Yes       939
Male    No       2625
        Yes       930
Name: count, dtype: int64

In [6]:
df = df.drop(columns='customerID') #удаляем строку ID так, как нам она не понадобится в обучении моделей
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [7]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors = 'coerce') #Меняем тип данных нам float64
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [9]:
# я считаю, что замена на 0 нам даст выбросы в данных из-за которых такие модели как XGBoost будут обучаться неправильно, тем самым мы меняем на медианное значение
total_charges_median = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(total_charges_median)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null   object 


In [10]:
le = LabelEncoder() # кодируем типы данных да/нет в 1/0
df['Churn'] = le.fit_transform(df['Churn'])
df['gender'] = le.fit_transform(df['gender'])
df['Partner'] = le.fit_transform(df['Partner'])
df['Dependents'] = le.fit_transform(df['Dependents'])
df['PhoneService'] = le.fit_transform(df['PhoneService'])
df['StreamingMovies'] = le.fit_transform(df['StreamingMovies'])
df['PaperlessBilling'] = le.fit_transform(df['PaperlessBilling'])



In [11]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,No phone service,DSL,No,Yes,No,No,No,0,Month-to-month,1,Electronic check,29.85,29.85,0
1,1,0,0,0,34,1,No,DSL,Yes,No,Yes,No,No,0,One year,0,Mailed check,56.95,1889.50,0
2,1,0,0,0,2,1,No,DSL,Yes,Yes,No,No,No,0,Month-to-month,1,Mailed check,53.85,108.15,1
3,1,0,0,0,45,0,No phone service,DSL,Yes,No,Yes,Yes,No,0,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,0,0,0,0,2,1,No,Fiber optic,No,No,No,No,No,0,Month-to-month,1,Electronic check,70.70,151.65,1


In [12]:
# так как тут мы имеем не бинарные данные то пользуемся one-hot encoder
df_encoded = pd.get_dummies(data=df, 
                            columns=['MultipleLines', 'Contract', 'PaymentMethod', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport'], 
                            drop_first=True, dtype=int)
df_encoded.head()
df_encoded['StreamingTV'] = le.fit_transform(df_encoded['StreamingTV'])
df_encoded.head()
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 29 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7043 non-null   int64  
 1   SeniorCitizen                          7043 non-null   int64  
 2   Partner                                7043 non-null   int64  
 3   Dependents                             7043 non-null   int64  
 4   tenure                                 7043 non-null   int64  
 5   PhoneService                           7043 non-null   int64  
 6   StreamingTV                            7043 non-null   int64  
 7   StreamingMovies                        7043 non-null   int64  
 8   PaperlessBilling                       7043 non-null   int64  
 9   MonthlyCharges                         7043 non-null   float64
 10  TotalCharges                           7043 non-null   float64
 11  Chur

In [13]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 29 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7043 non-null   int64  
 1   SeniorCitizen                          7043 non-null   int64  
 2   Partner                                7043 non-null   int64  
 3   Dependents                             7043 non-null   int64  
 4   tenure                                 7043 non-null   int64  
 5   PhoneService                           7043 non-null   int64  
 6   StreamingTV                            7043 non-null   int64  
 7   StreamingMovies                        7043 non-null   int64  
 8   PaperlessBilling                       7043 non-null   int64  
 9   MonthlyCharges                         7043 non-null   float64
 10  TotalCharges                           7043 non-null   float64
 11  Chur

In [14]:
df_encoded['StreamingTV'].head()

0    0
1    0
2    0
3    0
4    0
Name: StreamingTV, dtype: int64

In [15]:
results = {}

In [16]:
X = df_encoded.drop(columns='Churn') # Обучаем модель logisticregression
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
values_to_scale_train = ['tenure', 'MonthlyCharges', 'TotalCharges']

scaler = StandardScaler() # в scaler пишем
X_train_scaled = scaler.fit_transform(X_train[values_to_scale_train])
X_test_scaled = scaler.transform(X_test[values_to_scale_train])

X_train_part = X_train.drop(columns=values_to_scale_train)
X_test_part = X_test.drop(columns=values_to_scale_train)

X_train_final = pd.concat([
    X_train_part.reset_index(drop=True),
    pd.DataFrame(X_train_scaled, columns=values_to_scale_train)
], axis=1)

X_test_final = pd.concat([
    X_test_part.reset_index(drop=True),
    pd.DataFrame(X_test_scaled, columns=values_to_scale_train)
], axis=1)

lr = LogisticRegression(class_weight='balanced', max_iter=1000)
lr.fit(X_train_final, y_train)

y_proba = lr.predict_proba(X_test_final)[:, 1]
y_pred = (y_proba > 0.3).astype(int)

results['LR'] = (accuracy_score(y_test, y_pred), recall_score(y_test, y_pred), f1_score(y_test, y_pred), roc_auc_score(y_test, y_proba))

In [17]:
X = df_encoded.drop(columns='Churn')
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

rf = RandomForestClassifier(class_weight='balanced', n_estimators=300, max_depth=10)
rf.fit(X_train, y_train)

y_pred_proba = rf.predict_proba(X_test)[:, 1]
y_pred_rf = (y_pred_proba > 0.3).astype(int)

results['RF'] = (accuracy_score(y_test, y_pred_rf), recall_score(y_test, y_pred_rf), f1_score(y_test, y_pred_rf), roc_auc_score(y_test, y_pred_proba))

In [18]:
X = df_encoded.drop(columns='Churn')
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    learning_rate=0.05,
    max_depth = 4,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

y_proba_xgb = model.predict_proba(X_test)[:, 1]
y_pred_xgb = (y_proba_xgb > 0.3).astype(int)
results['XGB'] = (accuracy_score(y_test, y_pred_xgb), recall_score(y_test, y_pred_xgb), f1_score(y_test, y_pred_xgb), roc_auc_score(y_test, y_proba_xgb))

In [19]:
X = df_encoded.drop(columns='Churn')
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model_cat = CatBoostClassifier(
    auto_class_weights='Balanced',
    iterations=300,
    learning_rate=0.05,
    depth=4,
    random_state=42,
    verbose=0
)
model_cat.fit(X_train, y_train)
proba_cat = model_cat.predict_proba(X_test)[:, 1]
predict_cat = (proba_cat > 0.3).astype(int)
results['CAT'] = (accuracy_score(y_test, predict_cat), recall_score(y_test, predict_cat), f1_score(y_test, predict_cat), roc_auc_score(y_test, proba_cat))

In [20]:
X = df_encoded.drop(columns='Churn')
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = lgb.LGBMClassifier(class_weight='balanced',
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state = 42,
    verbose=-1
)
model.fit(X_train, y_train)

y_proba_lgb = model.predict_proba(X_test)[:, 1]
y_pred_lgb = (y_proba_lgb > 0.3).astype(int)
results['LGB'] = (accuracy_score(y_test, y_pred_lgb), recall_score(y_test, y_pred_lgb), f1_score(y_test, y_pred_lgb), roc_auc_score(y_test, y_proba_lgb))

In [21]:
results_df = pd.DataFrame(results, index=['Accuracy', 'Recall', 'F1', 'ROC-AUC']).T
print(results_df.sort_values('F1', ascending=False))

     Accuracy    Recall        F1   ROC-AUC
RF   0.705632  0.868093  0.610276  0.838592
LGB  0.699479  0.855615  0.601881  0.830155
CAT  0.675816  0.905526  0.597296  0.846055
XGB  0.675816  0.887701  0.592504  0.835501
LR   0.650734  0.928699  0.585393  0.844426


In [22]:
from sklearn.metrics import precision_score
prec = precision_score(y_test, y_pred)
print(f"Точность (Precision) LR: {prec:.2%}. Ложных тревог: {(1-prec):.2%}")

prec_LR = precision_score(y_test, y_pred_rf)
print(f'Точность: (Precision) LR: {prec_LR:.2%}. Ложных тревог: {(1-prec_LR):.2%}')

Точность (Precision) LR: 42.74%. Ложных тревог: 57.26%
Точность: (Precision) LR: 47.05%. Ложных тревог: 52.95%


## Выводы по результатм моделей: Так как наша основная метрика Recall, потому что требуется найти действительно истинных людей которые уйдут и которым стоит звонить. Лучше всего себя показывает LogisticRegression - Recall(0.9287), но Accuracy самая низкая из всех - (0.6507) - как и F1 - (0.5854). Мы опустили порог до 0.3, чтобы модели чаще находили людей, которые уйдут (категория 1), тем самым модель хуже предсказывает тех кто не собирается уходить.
### Вопрос бизнесу: Какую модель мы решим оставить и что для нас важнее - мы можем находить 93% уходящих, но тогда придется звонить 57.26% лояльных клиентов (ложные тревоги). Или мы можем ловить 87% уходящих клиентов, и звонить 53.52% лоявльных клиентов? Что выбираем?